# 2 - Online Sales Prediction - Regression

<img src='https://ec.europa.eu/eurostat/documents/4187653/15566018/Rawpixel.com_Shutterstock_461355724_RV.jpg/3fb48174-9060-c447-a74e-3ba65ea7c703?version=1.0&t=1676467745417'>

Bu çalışmada e-ticaret ve dijital pazarlama verilerini kullanarak satış gelirini tahmin eden bir regression modeli geliştireceğim. Bu proje özellikle online satış performansını öngörmek için uygun bir örnek oluşturuyor.


## Akış

1. Veriyi yükleme
2. Veriyi inceleme
3. Veri temizleme
4. Feature engineering
5. Kategorik verileri sayısala çevirme
6. Regression modelleri kurma
7. Model sonuçlarını karşılaştırma
8. Sonuç


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Veriyi Yükleme


In [2]:
# Bu projede Kaggle'dan indirilen dataseti Colab üzerinden zip dosyası olarak açıp kullanacağım.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import zipfile

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/E-Commerce Marketing & Sales Revenue Prediction.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content')


In [5]:
import os
os.listdir('/content')[:50]


['.config', 'train.csv', 'drive', 'test.csv', 'sample_data']

## 2. Veriyi Okuma ve İnceleme


In [6]:
# Bu bölümde zip içinden çıkan gerçek csv dosyasını okuyup veriyi inceliyoruz.

In [7]:
file_path = '/content/train.csv'

df = pd.read_csv(file_path)
df.head()


,id,date,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,1,2011-12-05 11:31:00,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,2,2011-04-27 14:01:00,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,3,2010-11-09 15:20:00,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,4,2010-10-03 15:24:00,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,5,2011-10-14 09:28:00,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924


In [8]:
df.shape


(18000, 17)

In [9]:
df.columns


Index(['id', 'date', 'region', 'channel', 'product_category',
       'customer_segment', 'ad_spend', 'price', 'discount_rate',
       'market_reach', 'impressions', 'click_through_rate',
       'competition_index', 'seasonality_index', 'campaign_duration_days',
       'customer_lifetime_value', 'sales_revenue'],
      dtype='object')

In [10]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       18000 non-null  int64  
 1   date                     18000 non-null  object 
 2   region                   18000 non-null  object 
 3   channel                  18000 non-null  object 
 4   product_category         18000 non-null  object 
 5   customer_segment         18000 non-null  object 
 6   ad_spend                 17342 non-null  float64
 7   price                    18000 non-null  float64
 8   discount_rate            17245 non-null  float64
 9   market_reach             17314 non-null  float64
 10  impressions              18000 non-null  int64  
 11  click_through_rate       17287 non-null  float64
 12  competition_index        17293 non-null  float64
 13  seasonality_index        18000 non-null  float64
 14  campaign_duration_days

## 3. Veri Temizleme


In [11]:
# Bu bölümde tekrar eden satırları, boş verileri ve veri tiplerini kontrol edeceğim.


In [12]:
df.duplicated().sum()


np.int64(0)

In [13]:
df = df.drop_duplicates()


In [14]:
df.isnull().sum()


,0
id,0
date,0
region,0
channel,0
product_category,0
customer_segment,0
ad_spend,658
price,0
discount_rate,755
market_reach,686


In [15]:
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    df[col] = df[col].fillna(df[col].median())

df.isnull().sum()


,0
id,0
date,0
region,0
channel,0
product_category,0
customer_segment,0
ad_spend,0
price,0
discount_rate,0
market_reach,0


## 4. Feature Engineering


In [16]:
# Bu bölümde modeli gereksiz yere büyüten sütunları çıkarıp satış geliriyle daha doğrudan ilişkili sayısal sütunlarla devam edeceğim.


In [17]:
df.head()


,id,date,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,1,2011-12-05 11:31:00,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,2,2011-04-27 14:01:00,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,3,2010-11-09 15:20:00,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,4,2010-10-03 15:24:00,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,5,2011-10-14 09:28:00,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924


## 5. Kategorik Verileri Sayısala Çevirme


In [19]:
# Bu projede sadece sayısal ve doğrudan satış geliriyle ilişkili sütunlarla devam ediyorum.
# id - sadece kayıt numarası olduğu için çıkardım.
# date - zaman bilgisi içeriyor ama bu projede doğrudan kullanmıyorum.
# region - kategori sayısını artırdığı için çıkardım.
# channel - çok fazla dummy sütun üretip modeli yavaşlattığı için çıkardım.
# product_category - sütun sayısını gereksiz artırdığı için çıkardım.
# customer_segment - modeli şişirdiği için çıkardım.

df.head()


,id,date,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,1,2011-12-05 11:31:00,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,2,2011-04-27 14:01:00,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,3,2010-11-09 15:20:00,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,4,2010-10-03 15:24:00,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,5,2011-10-14 09:28:00,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924


## 6. Regression Modelleri


In [20]:
# Bu bölümde birkaç farklı regression modeli kurup sonuçları karşılaştıracağım.


In [23]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

# Drop non-numeric and unnecessary columns
x = df.drop(['sales_revenue', 'id', 'date', 'region', 'channel', 'product_category', 'customer_segment'], axis=1)
y = df['sales_revenue']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [24]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append([name, mse, r2])

results_df = pd.DataFrame(results, columns=['Model', 'MSE', 'R2'])
results_df


,Model,MSE,R2
0,Linear Regression,2408.722652,0.227723
1,Ridge Regression,2407.197335,0.228212
2,Lasso Regression,2568.759718,0.176412
3,Random Forest,383.850903,0.876931
4,Gradient Boosting,346.513591,0.888902
5,Extra Trees,383.743459,0.876965


## 7. Sonuçları Karşılaştırma


In [25]:
# Burada hangi modelin daha başarılı olduğuna bakacağım.


In [26]:
results_df.sort_values('R2', ascending=False)


,Model,MSE,R2
4,Gradient Boosting,346.513591,0.888902
5,Extra Trees,383.743459,0.876965
3,Random Forest,383.850903,0.876931
1,Ridge Regression,2407.197335,0.228212
0,Linear Regression,2408.722652,0.227723
2,Lasso Regression,2568.759718,0.176412


## Sonuç


E-ticaret ve dijital pazarlama verileri kullanılarak satış geliri tahmini yapan regression modelleri denendi. Elde edilen sonuçlara göre en başarılı model Gradient Boosting oldu ve model 0.8889 R2 skoru elde etti.